#API INJECTION

In [0]:
import requests
import json

# 1. Configura tus datos
RAPIDAPI_KEY = "97c98f8504msh6925ec83e8a4ddap1f029bjsn6fe082c87264"
url = "https://realtor16.p.rapidapi.com/search/forsale"

# 2. Parámetros de búsqueda
querystring = {
    "location": "Miami, FL",   # Ciudad de búsqueda
    "page": "1",
    "limit": "20",             # Cuántas propiedades traer
    "sort": "newest_listing"
}

headers = {
    "x-rapidapi-key": RAPIDAPI_KEY,
    "x-rapidapi-host": "realtor16.p.rapidapi.com"
}

# 3. Petición GET
response = requests.get(url, headers=headers, params=querystring, timeout=15)

print(f"Status Code: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    # La API suele devolver la lista en 'results' o en la raíz, en esta caso es 'properties'
    propiedades = data.get("properties", data) if isinstance(data, dict) else data
    print(f"¡Éxito! Propiedades obtenidas: {len(propiedades)}")
    
    if len(propiedades) > 0:
        print("\nEjemplo de la primera propiedad:")
        print(json.dumps(propiedades[1], indent=2))
else:
    print(f"Error {response.status_code}: {response.text}")





In [0]:
import requests
import json
import time
from datetime import datetime

RAPIDAPI_KEY = "97c98f8504msh6925ec83e8a4ddap1f029bjsn6fe082c87264"
BASE_URL = "https://realtor16.p.rapidapi.com/search/forsale"

HEADERS = {
    "x-rapidapi-key": RAPIDAPI_KEY,
    "x-rapidapi-host": "realtor16.p.rapidapi.com"
}

# 1. Definimos las ciudades clave que queremos monitorear
TARGET_CITIES = ["Miami, FL",
    "Orlando, FL",
    "Austin, TX"]

# Parámetros de recolección
MAX_PAGES_PER_CITY = 2  # 2 páginas por ciudad para cuidar tu cuota gratuita
LIMIT_PER_PAGE = 20     # 20 casas por página = 40 casas por ciudad = 360 casas
all_extracted_records = []

for city in TARGET_CITIES:
    print(f"--- Iniciando extracción para: {city} ---")
    
    for page in range(1, MAX_PAGES_PER_CITY + 1):
        params = {
            "location": city,
            "page": str(page),
            "limit": str(LIMIT_PER_PAGE),
            "sort": "newest_listing"  # Traer las propiedades más recientes
        }
        
        try:
            response = requests.get(BASE_URL, headers=HEADERS, params=params, timeout=15)
            
            if response.status_code == 200:
                data = response.json()
                # Extraer la lista de resultados según la estructura de la API
                results = data.get("results", data) if isinstance(data, dict) else data
                
                if not results:
                    print(f"No hay más datos para {city} en la página {page}.")
                    break
                
                print(f"  Página {page}: {len(results)} propiedades extraídas.")
                
            elif response.status_code == 429:
                print("Alerta: Rate limit alcanzado. Pausando peticiones...")
                time.sleep(5)
            else:
                print(f"Error {response.status_code} en {city}, pág {page}: {response.text}")
                break
                
        except Exception as e:
            print(f"Excepción en la llamada: {e}")
            break
            
        # Buena práctica: pausa breve entre llamadas para no saturar la API
        time.sleep(1)

print(f"\nExtracción finalizada. Total de propiedades recolectadas: {len(all_extracted_records)}")

> ###Unnest

In [0]:
# Desanidar las listas anidadas dentro de cada propiedad usando for loops
registros = []

for prop in propiedades:
    # Extraer campos escalares y de dicts anidados
    property_id = prop.get("property_id", "")
    listing_id = prop.get("listing_id", "")
    list_price = prop.get("list_price", None)
    status = prop.get("status", "")
    list_date = prop.get("list_date", "")
    permalink = prop.get("permalink", "")

    # Desanidar location.address
    address = prop.get("location", {}).get("address", {})
    city = address.get("city", "")
    state = address.get("state", "")
    postal_code = address.get("postal_code", "")
    line = address.get("line", "")
    lat = address.get("coordinate", {}).get("lat", None)
    lon = address.get("coordinate", {}).get("lon", None)

    # Desanidar description
    desc = prop.get("description", {})
    beds = desc.get("beds", None)
    baths = desc.get("baths_consolidated", "")
    sqft = desc.get("sqft", None)
    prop_type = desc.get("type", "")
    sub_type = desc.get("sub_type", "")

    # --- Desanidar branding (lista) ---
    for brand in prop.get("branding", []):
        registros.append({
            "property_id": property_id,
            "listing_id": listing_id,
            "list_price": list_price,
            "status": status,
            "list_date": list_date,
            "city": city,
            "state": state,
            "postal_code": postal_code,
            "address_line": line,
            "lat": lat,
            "lon": lon,
            "beds": beds,
            "baths": baths,
            "sqft": sqft,
            "prop_type": prop_type,
            "sub_type": sub_type,
            "nested_field": "branding",
            "nested_name": brand.get("name", ""),
            "nested_type": brand.get("type", ""),
            "nested_photo": brand.get("photo", []),
            "permalink": permalink
        })

    # --- Desanidar photos (lista) ---
    for photo in prop.get("photos", []):
        registros.append({
            "property_id": property_id,
            "listing_id": listing_id,
            "list_price": list_price,
            "status": status,
            "list_date": list_date,
            "city": city,
            "state": state,
            "postal_code": postal_code,
            "address_line": line,
            "lat": lat,
            "lon": lon,
            "beds": beds,
            "baths": baths,
            "sqft": sqft,
            "prop_type": prop_type,
            "sub_type": sub_type,
            "nested_field": "photos",
            "nested_name": "",
            "nested_type": "",
            "nested_photo": photo.get("href", ""),
            "permalink": permalink
        })

    # --- Desanidar advertisers (lista) ---
    for adv in prop.get("advertisers", []):
        registros.append({
            "property_id": property_id,
            "listing_id": listing_id,
            "list_price": list_price,
            "status": status,
            "list_date": list_date,
            "city": city,
            "state": state,
            "postal_code": postal_code,
            "address_line": line,
            "lat": lat,
            "lon": lon,
            "beds": beds,
            "baths": baths,
            "sqft": sqft,
            "prop_type": prop_type,
            "sub_type": sub_type,
            "nested_field": "advertisers",
            "nested_name": "",
            "nested_type": adv.get("type", ""),
            "nested_photo": "",
            "permalink": permalink
        })

print(f"Total de registros desanidados: {len(registros)}")
print(f"Ejemplo del primer registro:\n{json.dumps(registros[0], indent=2, default=str)}")

###Set the current time, and convert to a data frame.

In [0]:
# Normalizar listas a strings para evitar conflictos de tipo en Spark
for r in registros:
    for k, v in r.items():
        if isinstance(v, list):
            r[k] = str(v) if v else ""

df_realtor = spark.createDataFrame(registros)
df_realtor_bronze = df_realtor.withColumn("ingesta_timestamp", current_timestamp())
df_realtor_bronze.write.format("delta").mode("overwrite").saveAsTable("realtor.landing.realtor_api")

spark.table("realtor.landing.realtor_api").show(10)